In [3]:
import pandas as pd
import numpy as np
import re

nrc = pd.read_csv('NRC-VAD-Lexicon-v2.1.txt', sep='\t')
nrc = nrc.rename(columns=lambda column: column.strip().lower())

word_column = 'word' if 'word' in nrc.columns else 'term' if 'term' in nrc.columns else None
required_columns = {'valence', 'arousal'}
missing_columns = required_columns.difference(nrc.columns)
if word_column is None:
    missing_columns.add('word/term')
if missing_columns:
    raise KeyError(f"Missing expected columns: {sorted(missing_columns)}. Found: {list(nrc.columns)}")

nrc['word'] = nrc[word_column].str.lower()

vad_lookup = nrc.set_index('word')[['valence', 'arousal']].to_dict('index')

In [16]:
def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

def sentence_to_va(sentence):
    tokens = tokenize(sentence)
    vals = [vad_lookup[t] for t in tokens if t in vad_lookup]

    if not vals:
        return None

    return {
        'valence': np.mean([value['valence'] for value in vals]),
        'arousal': np.mean([value['arousal'] for value in vals]),
    }

In [11]:
def lyrics_to_df(lyrics, song_id):
    lines = [l.strip() for l in lyrics.split('\n') if l.strip()]
    rows = []

    for i, line in enumerate(lines):
        va = sentence_to_va(line)
        if va:
            rows.append({
                'song_id': song_id,
                'sentence_idx': i,
                'valence': va['valence'],
                'arousal': va['arousal']
            })

    return pd.DataFrame(rows)

In [7]:
def compute_emotion_dynamics(song_df):
    v = song_df['valence'].values
    a = song_df['arousal'].values

    if len(v) < 2:
        return None
    
    return {
        "mean_v": np.mean(v),
        "mean_a": np.mean(a),
        "std_v": np.std(v),
        "std_a": np.std(a),
        "instability_v": np.mean(np.abs(np.diff(v))),
        "instability_a": np.mean(np.abs(np.diff(a))),
        "inertia_v": np.corrcoef(v[:-1], v[1:])[0, 1],
        "inertia_a": np.corrcoef(a[:-1], a[1:])[0, 1],
        "slope_v": np.polyfit(range(len(v)), v, 1)[0],
        "slope_a": np.polyfit(range(len(a)), a, 1)[0],
    }

In [12]:
df = pd.read_csv('User-Songs-Music-Group.xls')
df = df.rename(columns=lambda column: column.strip().lower())

lyric_column = 'lyrics' if 'lyrics' in df.columns else 'lyric' if 'lyric' in df.columns else None
label_column = 'label' if 'label' in df.columns else 'disorder' if 'disorder' in df.columns else 'sentiment_direction' if 'sentiment_direction' in df.columns else None

if 'song_id' not in df.columns:
    if {'artist', 'title'}.issubset(df.columns):
        df['song_id'] = df['artist'].fillna('').str.strip() + ' - ' + df['title'].fillna('').str.strip()
    elif 'title' in df.columns:
        df['song_id'] = df['title'].fillna('').str.strip()
    else:
        df['song_id'] = df.index.astype(str)

missing_columns = []
if lyric_column is None:
    missing_columns.append('lyrics/lyric')
if label_column is None:
    missing_columns.append('label/disorder/sentiment_direction')
if missing_columns:
    raise KeyError(f"Missing expected columns: {missing_columns}. Found: {list(df.columns)}")

df['lyrics'] = df[lyric_column].fillna('').astype(str)
df['label'] = df[label_column].fillna('unknown')

In [27]:
all_sentences = []
selected_labels = {'depression', 'control'}
df_subset = df[df['label'].astype(str).str.lower().isin(selected_labels)].copy()

if df_subset.empty:
    raise ValueError("No songs found for labels: depression/control")

for row in df_subset.itertuples(index=False):
    temp = lyrics_to_df(row.lyrics, row.song_id)
    if temp.empty:
        continue
    temp['label'] = str(row.label).lower()
    all_sentences.append(temp)

if not all_sentences:
    raise ValueError('No sentence-level VAD scores were produced from depression/control songs.')

df_sentences = pd.concat(all_sentences, ignore_index=True)
df_sentences

,song_id,sentence_idx,valence,arousal,label
0,Quadeca - Burnin Bridges / Long Day (feat. IDK),0,0.055333,-0.037333,depression
1,Quadeca - Burnin Bridges / Long Day (feat. IDK),1,-0.422000,-0.338000,depression
2,Quadeca - Burnin Bridges / Long Day (feat. IDK),3,-0.228000,0.598000,depression
3,Quadeca - Burnin Bridges / Long Day (feat. IDK),4,0.213000,0.367000,depression
4,Quadeca - Burnin Bridges / Long Day (feat. IDK),5,0.458000,-0.260000,depression
...,...,...,...,...,...
6857539,Bonnie McKee - American Girl,57,0.000000,-0.053500,control
6857540,Bonnie McKee - American Girl,58,0.000000,0.000000,control
6857541,Bonnie McKee - American Girl,59,0.108000,0.027600,control
6857542,Bonnie McKee - American Girl,60,-0.010500,-0.061000,control


In [28]:
results = []

for song_id, group in df_sentences.groupby('song_id'):
    metrics = compute_emotion_dynamics(group)
    if metrics:
        metrics['song_id'] = song_id
        metrics['label'] = group['label'].iloc[0]
        results.append(metrics)

results_df = pd.DataFrame(results)

/home/filipepimentel/quantum/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/home/filipepimentel/quantum/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/home/filipepimentel/quantum/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/home/filipepimentel/quantum/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/filipepimentel/quantum/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [32]:
print(results_df.groupby('label').mean(numeric_only=True))

              mean_v    mean_a     std_v     std_a  instability_v  \
label                                                               
control     0.132777 -0.061247  0.205104  0.185344       0.217890   
depression  0.134152 -0.069066  0.205812  0.184022       0.219019   

            instability_a  inertia_v  inertia_a   slope_v   slope_a  
label                                                                
control          0.196767   0.039383   0.030719 -0.000265  0.000937  
depression       0.196175   0.034143   0.023145 -0.000390  0.001060  
